# Patch risk triage

**Recipe 31** · Level 3 (Intermediate) · Decision type: `Noul`

Assess a small patch for separate authentication, input-validation, and data-loss concerns to prioritize human review using explicit rules.

Sources:

- S02: [TypeSafe AI: Primitives (Questions)](https://docs.typesafe.ai/primitives)
- S03: [TypeSafe AI: Confidence](https://docs.typesafe.ai/confidence)
- S04: [TypeSafe AI: How to build with TypeSafe](https://docs.typesafe.ai/concepts/how-to-build-with-system-one)
- S05: [TypeSafe AI: Example use cases](https://docs.typesafe.ai/concepts/use-case-map)

## What you will build

A review-priority triage over 40 fabricated patches: small, schematic diff excerpts, each touching an invented service's authentication check, its handling of one external field, and a cleanup job's delete of one table. Jev answers three independent `Noul` propositions about the same patch in one request: does it weaken or bypass an authentication check, does it leave external input unvalidated before a sensitive operation, and does it risk permanent data loss. The fixtures cover all eight combinations the three propositions can take together, so every pair of them is exercised in all four of its own combinations -- each alone, both at once, and neither -- without a fixture group dedicated to any one pair. Three further patches, one per concern, genuinely have that one concern but describe it next to a reassuring, specific-sounding comment that does not actually show a safeguard -- the realistic way a reviewer misses a real problem by trusting a comment instead of the code. Python composes the three answers into one of three review priorities -- `urgent`, `elevated` or `routine` -- or an explicit `review` outcome, in a single tested function, using a business threshold chosen per concern and one confidence gate shared by all three. The evaluation reports precision and recall for each concern on its own, and, at the level of the whole patch, how often the composed review priority matches the patch's gold combination of concerns.

**This recipe prioritises which patches a human reviewer looks at first. It is not a security scanner**: it finds nothing Python and Jev were not each explicitly asked to check, and a `routine` priority is not a statement that a patch is safe -- it only means none of these three particular concerns was flagged confidently enough to say otherwise.

## Setup and run mode

The notebook runs from its own folder with the standard library and `jev_cookbook` only. `get_backend(fixtures=...)` replays the 40 stored answers in `fixtures/responses.json` offline; setting `JEV_COOKBOOK_LIVE=1` (see the README) swaps in live calls to the same three question definitions and changes nothing else. `run_header` states which mode ran, and every number below carries that mode with it. Each patch needs exactly one request (all three concerns are independent questions over the same diff, so they are asked together), so this notebook's request count and its decision count are not the same thing: 40 requests produce 120 label decisions, three per patch.

In [ ]:
from jev_cookbook import get_backend, load_helpers
from jev_cookbook.evaluation import (
    accuracy,
    confusion_matrix,
    evaluate_outcomes,
    evaluate_selective,
    evaluate_threshold,
    pool_label_decisions,
    pool_label_outcomes,
    selective_curve,
    threshold_sweep,
)
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path
from jev_cookbook.simulation import ReviewQueue
from jev_cookbook.style import (
    apply_style,
    plot_answer_probabilities,
    plot_confusion_matrix,
    plot_risk_coverage,
    plot_threshold_sweep,
    run_header,
    show_answer,
)

apply_style()
helpers = load_helpers()  # this recipe's helpers.py, loaded by path
examples = load_inputs()
labels = load_labels()
backend = get_backend(fixtures=responses_path())
questions = helpers.build_questions()
# N for the header is the number of examples the metrics are about: validation and test. The
# two demo patches are shown below but never scored.
scored = [e for e in examples if e.split not in ("train", "demo")]

offline = backend.mode in ("synthetic", "scripted")
check = " (a pipeline check, not a Jev result)" if offline else ""
# CONTRIBUTING.md section 2: a number from the validation split is a selection step, not a
# result. A synthetic or scripted run carries both labels on a validation number.
selection = " (a selection step, not a reported result)"

# Every patch needs at most one request; this cache makes sure a patch shown individually and
# later scored in its split is decided once, so the whole notebook makes exactly one request
# per fixture patch in live mode, never more.
decided = {}


def decide(example):
    if example.id not in decided:
        decided[example.id] = backend.decide(helpers.build_state(example.fields), questions)
    return decided[example.id]


run_header(31, "Patch risk triage", backend=backend, n_examples=len(scored))

## The state

The state is everything Jev sees, and Python builds it. Each example in `fixtures/inputs.jsonl` carries `fields`: a `diff`, the full fabricated patch excerpt. `build_state` passes the diff on unchanged -- a patch submitted for review is read in full by whoever triages it, so there is nothing for Python to trim before Jev sees the same text a human reviewer would. The example id never passes through the model either; it stays in Python and is attached to the outcome at the end. Every file, function, package and ticket name in every patch below is fabricated ([CONTRIBUTING.md](../../CONTRIBUTING.md) section 5); no hunk is complete or specific enough to apply or execute as a real patch.

In [ ]:
import json

demo = {e.id: e for e in examples if e.split == "demo"}
routine_demo = demo["d01"]
print("Python has:")
print(json.dumps(routine_demo.fields, indent=2))
state = helpers.build_state(routine_demo.fields)
print("Jev sees:")
print(json.dumps(state, indent=2))

## The questions

Three independent propositions describe the same patch, so `backend.decide` sends them together in one request: TypeSafe's primitives page ([S02](https://docs.typesafe.ai/primitives)) says independent questions over the same state belong in one request. Each is written as a statement that is true or false, as [CONTRIBUTING.md](../../CONTRIBUTING.md) requires of a `Noul`, not as a question: `authentication_concern` is true when the diff removes, disables or bypasses a check on who a caller is or what they may do; `input_validation_concern` is true when it leaves external input unvalidated before a sensitive operation; `data_loss_concern` is true when it risks permanent loss of stored data. They stay three separate `Noul` propositions rather than one `Choice` because a patch can independently carry or lack each kind of concern -- a `Choice` would force Jev to adjudicate the combination itself, when the combination is exactly what Python decomposes and composes below. Each name ends `_concern` so that, in prose below, a proposition is never confused with the `validation` fixture split of the same word.

In [ ]:
for name, question in questions.items():
    print(name)
    print(" ", question.instructions)
    for side, description in question.criteria.items():
        print(f"    {side}: {description}")

## One answer up close

Before any aggregate, look at four typed patches: a clean patch with none of the three concerns, a patch with exactly one concern (authentication alone), a patch with exactly one concern of the kind this recipe's explicit rule treats as urgent on its own (data loss alone), and a patch whose reading is close enough to the line that Python is not confident enough to trust either way. Each answer holds just `noul`, the probability that its proposition is true, and a `provenance` line; a `Noul` has no separate [confidence](../../docs/glossary.md#confidence) field in the API response at all (S02, S03), unlike `Choice` and `Score`. All four come from `demo` or `validation`, never from `test`, which is reported once further down.

In [ ]:
routine_answer = decide(routine_demo)
print(routine_demo.fields["diff"])
show_answer(routine_answer["authentication_concern"])
show_answer(routine_answer["input_validation_concern"])
show_answer(routine_answer["data_loss_concern"])
plot_answer_probabilities(
    routine_answer["authentication_concern"],
    title="authentication_concern, a patch with none of the three concerns flagged",
)

Every stored probability reads low: authentication 0.04, input validation 0.05, data loss 0.03. This is the clean case the rule below reports `routine` for: nothing flagged, nothing to review.

In [ ]:
elevated_example = next(e for e in examples if e.id == "v03")
elevated_answer = decide(elevated_example)
print(elevated_example.fields["diff"])
show_answer(elevated_answer["authentication_concern"])
show_answer(elevated_answer["input_validation_concern"])
show_answer(elevated_answer["data_loss_concern"])

Authentication reads high (0.97): the diff removes the permission check and only leaves a comment behind. Input validation and data loss both read low (0.02 each): neither hunk touches how the patch validates input or deletes data. One concern alone, with the other two clean, is the `elevated` case below -- worth a reviewer's attention sooner than `routine`, but on its own not the combination this recipe's rule treats as `urgent`.

In [ ]:
urgent_example = next(e for e in examples if e.id == "v07")
urgent_answer = decide(urgent_example)
print(urgent_example.fields["diff"])
show_answer(urgent_answer["authentication_concern"])
show_answer(urgent_answer["input_validation_concern"])
show_answer(urgent_answer["data_loss_concern"])

Data loss reads high (0.91) with authentication and input validation both low (0.08 each): the archive step before the delete is gone, and nothing else about this patch is concerning. "Python's part" below treats a flagged data-loss concern as `urgent` on its own, unlike authentication or input validation alone, which read only `elevated` -- a stored data item, once deleted without a safeguard, cannot be recovered by a later review the way a missed permission check still could be before anyone exploits it.

In [ ]:
review_demo = demo["d02"]
review_answer = decide(review_demo)
print(review_demo.fields["diff"])
show_answer(review_answer["authentication_concern"])
show_answer(review_answer["input_validation_concern"])
show_answer(review_answer["data_loss_concern"])

Authentication reads 0.62 -- above where "Python's part" below ends up freezing the business threshold for a flagged concern, so the raw reading alone would call this patch's authentication check compromised -- but its stored [confidence](../../docs/glossary.md#confidence), `|2(0.62) - 1| = 0.24`, is low enough that the [confidence gate](../../docs/glossary.md#confidence-gate) below sends the whole patch to [review](../../docs/glossary.md#review) rather than act on a reading this close to the line.

## Python's part

Jev supplies three probabilities; everything that happens with them is code. `helpers.decide_concerns` turns each one into a [business threshold](../../docs/glossary.md#business-threshold) tag -- `authentication_concern >= t_authentication`, and likewise for the other two -- and checks it against one shared [confidence gate](../../docs/glossary.md#confidence-gate), `noul_confidence(noul) >= gate`, for any of the three. `helpers.review_priority` then composes the three label decisions into one review priority: a patch with any label gated goes to an explicit [review](../../docs/glossary.md#review) outcome, whichever way its tags read; otherwise `helpers.priority_from_concerns` turns the set of flagged concerns into `urgent` (a data-loss concern on its own, or authentication together with input validation), `elevated` (exactly one of authentication or input validation, no data loss), or `routine` (none flagged). `tests/test_helpers.py` checks every branch directly, every one of the eight possible concern combinations, and -- on real, committed `test` patches -- that flipping any one stored decision on its own, holding the other two fixed, changes the composed priority.

Each per-concern threshold is chosen independently, by F1 (`helpers.choose_label_threshold`), and the one shared gate is chosen afterwards from all three concerns' pooled decisions, by the minimum-coverage objective `helpers.choose_gate` (`min_coverage=`, [`helpers.MIN_COVERAGE`](helpers.py)) rather than a 1.0 accuracy target: a 1.0 target would force validation accuracy to 1.0 by construction, which is not a property this recipe wants to report. All four choices are made on `validation`, then frozen.

In [ ]:
val_examples = [e for e in examples if e.split == "validation"]
val_answers = {e.id: decide(e) for e in val_examples}
val_noul = {
    label: [val_answers[e.id][label].noul for e in val_examples] for label in helpers.LABELS
}
val_gold = {label: [labels[e.id][label] for e in val_examples] for label in helpers.LABELS}

thresholds = {
    label: helpers.choose_label_threshold(val_gold[label], val_noul[label])
    for label in helpers.LABELS
}
for label in helpers.LABELS:
    print(
        f"{label} business threshold, frozen from here on: {thresholds[label]:.3f}{selection}{check}"
    )

val_ids = [e.id for e in val_examples]
val_pooled_correct, val_pooled_confidence, val_pooled_keys = pool_label_decisions(
    val_ids, list(helpers.LABELS), val_gold, val_noul, thresholds
)
gate = helpers.choose_gate(val_pooled_correct, val_pooled_confidence)
print(f"confidence gate, frozen from here on: {gate:.3f}{selection}{check}")

for shown_id, shown in (
    ("d01", routine_demo),
    ("v03", elevated_example),
    ("v07", urgent_example),
    ("d02", review_demo),
):
    answer = decide(shown)
    decisions = helpers.decide_concerns(answer, thresholds, gate)
    result = helpers.review_priority(decisions)
    noul_text = ", ".join(f"{label} {answer[label].noul:.2f}" for label in helpers.LABELS)
    print(f"{shown_id}: {noul_text} -> {result.priority} ({result.reason})")

### Why one shared gate, and its blind spot

A multi-label `Noul` rule's three-path pattern -- answer true, answer false, or send a patch to a person -- needs every (patch, concern) decision's correctness and confidence pooled before a [coverage](../../docs/glossary.md#coverage) and [risk](../../docs/glossary.md#risk) curve can be drawn over all three concerns at once; `pool_label_decisions` does that pooling, and `selective_curve` is the sanity check that confidence actually separates right answers from wrong ones on this data, shown next rather than asserted, as a single shared gate over three different propositions. `|2p - 1|` measures distance from an even split, not distance from whichever business threshold a concern happens to use: a concern whose own threshold sits near 0.5 can read as confidently right even very close to it, because `noul_confidence` is never told where that threshold is. The gate chosen here is structurally blind to exactly that shape of error, on any of the three.

In [ ]:
val_curve = selective_curve(val_pooled_correct, val_pooled_confidence)
print(f"pooled, {len(val_pooled_correct)} label decisions{selection}{check}")
print(
    f"ungated accuracy: {sum(val_pooled_correct) / len(val_pooled_correct):.4f}{selection}{check}"
)
for t, cov, acc, risk in zip(
    val_curve.thresholds, val_curve.coverage, val_curve.accuracy, val_curve.risk, strict=True
):
    marker = " <- frozen" if abs(t - gate) < 1e-9 else ""
    print(
        f"  gate >= {t:.3f}: coverage {cov:.4f}, accuracy {acc:.4f}, "
        f"risk {risk:.4f}{marker}{selection}{check}"
    )
plot_risk_coverage(
    val_curve,
    reference_risk=1 - sum(val_pooled_correct) / len(val_pooled_correct),
    title=f"Validation, pooled label decisions{check}",
)

`jev_cookbook.simulation.ReviewQueue` is the shared primitive for the `review` outcome ([docs/simulation.md](../../docs/simulation.md)): `submit` records an item, the reason it was sent, and the typed answer behind that reason, and nothing in it executes anything. `ReviewQueue` itself keeps submission order, oldest first; it does not reorder anything on its own, so a queue that reads sorted is sorted once, before every item is submitted, by `helpers.review_sort_key` -- the lowest of the three concerns' own confidence, so the patch Python is least sure about surfaces first. A preview on the four patches shown above, with the full item a reviewer would actually see:

In [ ]:
preview_items = []
for shown_id, shown in (
    ("d01", routine_demo),
    ("v03", elevated_example),
    ("v07", urgent_example),
    ("d02", review_demo),
):
    answer = decide(shown)
    decisions = helpers.decide_concerns(answer, thresholds, gate)
    result = helpers.review_priority(decisions)
    if result.priority == helpers.REVIEW:
        preview_items.append((shown_id, shown, result))

preview_items.sort(key=lambda row: (helpers.review_sort_key(row[2]), row[0]))
print(f"preview: {len(preview_items)} of these four would be queued{selection}{check}")
for shown_id, shown, result in preview_items:
    item = helpers.queue_item(shown_id, shown.fields, result)
    print(f"  {item['id']}: {result.reason}, sort key {helpers.review_sort_key(result):.3f}")
    for label in helpers.LABELS:
        print(
            f"    {label}: noul {item[label]['noul']:.2f}, confidence {item[label]['confidence']:.4f}"
        )

## Evaluation

Both the three business thresholds and the shared confidence gate were chosen and frozen above, on `validation`. This section reports, per concern, its own [precision](../../docs/glossary.md#precision) and [recall](../../docs/glossary.md#recall) against its threshold alone (no gate); at task level, the counts `helpers.review_priority` actually returns and [`evaluate_outcomes`](../../docs/evaluation.md)'s [coverage](../../docs/glossary.md#coverage), [accuracy](../../docs/glossary.md#accuracy) and [risk](../../docs/glossary.md#risk) over the composed priority; a confusion matrix of the composed priority against gold; and the use case's own measure -- how often the composed review priority matches the patch's gold combination of concerns exactly. `test` is used once, after every choice above is frozen.

In [ ]:
val_results = {
    e.id: helpers.review_priority(helpers.decide_concerns(decide(e), thresholds, gate))
    for e in val_examples
}
val_counts = {helpers.ROUTINE: 0, helpers.ELEVATED: 0, helpers.URGENT: 0, helpers.REVIEW: 0}
for e in val_examples:
    val_counts[val_results[e.id].priority] += 1
val_accepted = [val_results[e.id].priority != helpers.REVIEW for e in val_examples]
val_correct = [val_results[e.id].priority == labels[e.id]["priority"] for e in val_examples]
val_outcomes = evaluate_outcomes(val_accepted, val_correct)

print(f"validation, {len(val_examples)} patches{selection}{check}")
print(
    f"routine: {val_counts[helpers.ROUTINE]}  elevated: {val_counts[helpers.ELEVATED]}  "
    f"urgent: {val_counts[helpers.URGENT]}  review: {val_counts[helpers.REVIEW]}{selection}{check}"
)
print(f"coverage: {val_outcomes.coverage:.4f}{selection}{check}")
print(f"accuracy among answered: {val_outcomes.accuracy:.4f}{selection}{check}")
print(f"risk among answered: {val_outcomes.risk:.4f}{selection}{check}")

[Precision](../../docs/glossary.md#precision) is the fraction of patches tagged as carrying a concern that genuinely do; [recall](../../docs/glossary.md#recall) is the fraction of patches that genuinely carry that concern the rule tags at all; [F1](../../docs/glossary.md#f1) is their harmonic mean. The sweep below is `data_loss_concern`'s own business rule alone -- the concern this recipe's explicit rule treats as urgent on its own -- over every threshold `helpers.choose_label_threshold` considered on `validation`, printed and then drawn.

In [ ]:
data_loss_sweep = threshold_sweep(
    val_gold[helpers.DATA_LOSS_CONCERN], val_noul[helpers.DATA_LOSS_CONCERN]
)
for point in data_loss_sweep:
    print(
        f"  threshold {point.threshold:.3f}: precision {point.precision:.4f}, "
        f"recall {point.recall:.4f}, f1 {point.f1:.4f}{selection}{check}"
    )
plot_threshold_sweep(
    data_loss_sweep,
    chosen=thresholds[helpers.DATA_LOSS_CONCERN],
    title=f"Validation, data_loss_concern, {len(val_examples)} patches{check}",
)

On `test`, each concern's own business rule, with no gate at all, is reported first with `evaluate_threshold`; then the composed, task-level priority `helpers.review_priority` actually returns, and its coverage, accuracy and risk from `evaluate_outcomes`, exactly as for `validation` above.

In [ ]:
test_examples = [e for e in examples if e.split == "test"]
test_answers = {e.id: decide(e) for e in test_examples}
test_noul = {
    label: [test_answers[e.id][label].noul for e in test_examples] for label in helpers.LABELS
}
test_gold = {label: [labels[e.id][label] for e in test_examples] for label in helpers.LABELS}

print(f"test, {len(test_examples)} patches, every threshold and the gate frozen{check}")
for label in helpers.LABELS:
    point = evaluate_threshold(test_gold[label], test_noul[label], thresholds[label])
    print(
        f"{label} alone: precision {point.precision:.4f}  recall {point.recall:.4f}  "
        f"f1 {point.f1:.4f}{check}"
    )

test_results = {
    e.id: helpers.review_priority(helpers.decide_concerns(decide(e), thresholds, gate))
    for e in test_examples
}
test_counts = {helpers.ROUTINE: 0, helpers.ELEVATED: 0, helpers.URGENT: 0, helpers.REVIEW: 0}
for e in test_examples:
    test_counts[test_results[e.id].priority] += 1
test_accepted = [test_results[e.id].priority != helpers.REVIEW for e in test_examples]
test_correct = [test_results[e.id].priority == labels[e.id]["priority"] for e in test_examples]
test_outcomes = evaluate_outcomes(test_accepted, test_correct)

print(
    f"routine: {test_counts[helpers.ROUTINE]}  elevated: {test_counts[helpers.ELEVATED]}  "
    f"urgent: {test_counts[helpers.URGENT]}  review: {test_counts[helpers.REVIEW]}{check}"
)
print(f"coverage: {test_outcomes.coverage:.4f}{check}")
print(f"accuracy among answered: {test_outcomes.accuracy:.4f}{check}")
print(f"risk among answered: {test_outcomes.risk:.4f}{check}")

Every one of the three concerns reads precision 1.0000, recall 0.8889 and F1 0.9412 on `test`: each misses exactly one patch, the one lookalike written for it (`t17` for authentication, `t18` for input validation, `t19` for data loss -- named again below). This symmetry is not guaranteed by the fixture design: the three concerns are independent (every one of the eight combinations appears in this set, so any one of them is free to read true or false on its own regardless of the other two), and `tests/test_helpers.py` proves each concern can change the composed priority by flipping a real stored decision. On `validation`, `input_validation_concern` reads precision 0.8889 rather than 1.0000: `v02`, a patch with none of the three concerns, carries a deliberately wrong, low-confidence `input_validation_concern` reading (0.63, just above the frozen threshold) that the rule tags as a concern anyway -- a false positive the business rule alone does not catch, though the confidence gate sends this particular patch to review regardless.

Among the 16 `test` patches `review_priority` actually answered (3 of the 19 are sent to review, counted above), a confusion matrix of gold against the composed priority names exactly which error is which.

In [ ]:
answered_examples = [e for e in test_examples if test_results[e.id].priority != helpers.REVIEW]
answered_gold = [labels[e.id]["priority"] for e in answered_examples]
answered_predicted = [test_results[e.id].priority for e in answered_examples]
matrix = confusion_matrix(
    answered_gold, answered_predicted, labels=[helpers.ROUTINE, helpers.ELEVATED, helpers.URGENT]
)
width = max(len(label) for label in matrix.labels)
leader_width = max(width + len("gold "), len("predicted ->"))
header = "  ".join(f"{label:>{width}s}" for label in matrix.labels)
print(f"Test confusion matrix (answered only), gold rows by predicted columns{check}:")
print(f"  {'predicted ->':<{leader_width}s}  {header}{check}")
for gold_label, row in zip(matrix.labels, matrix.matrix.tolist(), strict=True):
    counts = "  ".join(f"{count:{width}d}" for count in row)
    leader = f"gold {gold_label:<{width}s}"
    print(f"  {leader:<{leader_width}s}  {counts}{check}")
plot_confusion_matrix(matrix, title=f"Test split, answered only{check}")

The use case's own measure: among the same 16 answered patches, does the composed review priority match the patch's gold priority exactly -- the same quantity `evaluate_outcomes`'s `accuracy` reported above, computed directly here with `jev_cookbook.evaluation.accuracy` over the plain priority strings, as a second, independent way to read the same number.

In [ ]:
priority_agreement = accuracy(answered_gold, answered_predicted)
print(
    f"review priority matches gold exactly: {priority_agreement:.4f} "
    f"({len(answered_examples)} answered patches){check}"
)

The same pooled, per-concern view as `validation`'s own sanity check above, now on `test` with every cut-off frozen. The shared gate is this rule's only review branch -- nothing else sends a patch to review, or lets one through regardless of confidence -- so the confidence-only view (`evaluate_selective` over the pooled decisions) and the rule's own outcomes view (`evaluate_outcomes` over `pool_label_outcomes`) describe the same gate from the same granularity and agree on every field but `threshold` (undefined, not merely unreported, for the outcomes view); printed side by side rather than asserted, so a reader can see the agreement rather than take it on faith.

In [ ]:
test_ids = [e.id for e in test_examples]
test_pooled_correct, test_pooled_confidence, test_pooled_keys = pool_label_decisions(
    test_ids, list(helpers.LABELS), test_gold, test_noul, thresholds
)
pooled_selective = evaluate_selective(test_pooled_correct, test_pooled_confidence, gate)

test_decisions = {e.id: helpers.decide_concerns(decide(e), thresholds, gate) for e in test_examples}
accepted_by_label = {
    label: [not test_decisions[e.id][label].gated for e in test_examples]
    for label in helpers.LABELS
}
tag_by_label = {
    label: [test_decisions[e.id][label].tag for e in test_examples] for label in helpers.LABELS
}
pooled_accepted, pooled_correct, _pooled_keys = pool_label_outcomes(
    test_ids, list(helpers.LABELS), test_gold, accepted_by_label, tag_by_label
)
pooled_outcomes = evaluate_outcomes(pooled_accepted, pooled_correct)

print(f"pooled, {len(test_pooled_correct)} label decisions{check}")
print(f"ungated accuracy: {sum(test_pooled_correct) / len(test_pooled_correct):.4f}{check}")
print(
    f"confidence-only view:  coverage {pooled_selective.coverage:.4f}  "
    f"accuracy {pooled_selective.accuracy:.4f}  risk {pooled_selective.risk:.4f}{check}"
)
print(
    f"rule's own outcomes:   coverage {pooled_outcomes.coverage:.4f}  "
    f"accuracy {pooled_outcomes.accuracy:.4f}  risk {pooled_outcomes.risk:.4f}{check}"
)
print(f"the two agree because the confidence gate is this rule's only review branch{check}")

In [ ]:
print("Test patches the composed priority got wrong, and what the gate saw on each concern:")
for example in test_examples:
    result = test_results[example.id]
    gold_priority = labels[example.id]["priority"]
    if result.priority != helpers.REVIEW and result.priority != gold_priority:
        decisions = test_decisions[example.id]
        detail = ", ".join(
            f"{label} {decisions[label].noul:.2f} (confidence {decisions[label].confidence:.4f})"
            for label in helpers.LABELS
        )
        print(
            f"  {example.id}: gold {gold_priority} {sorted(helpers.gold_concern_set(labels[example.id]))}, "
            f"composed {result.priority} -- {detail}"
        )

`t17`, `t18` and `t19` each clear the frozen gate on all three concerns, printed in the cell above, and are each still wrong: every one is the lookalike written for its own concern -- a patch whose diff genuinely carries that concern, described next to a reassuring, specific-sounding comment (a ticket number, a claim that a gateway or the platform already handles it) that the diff itself does not actually show. This is exactly the blind spot named above: `noul_confidence` measures distance from 0.5, never distance from the frozen threshold, so a concern whose business threshold sits near 0.5 can read as confidently right even when it is simply wrong about the comment. A shared gate over three concerns cannot be told where any one concern's own threshold sits without ceasing to be one shared number, so this is a property of pooling the gate at all, not a bug to tune away with a different gate value -- and it is also the central lesson this recipe teaches: a reviewer (or a model) can be fooled by a comment that claims a safeguard the code does not have, which is exactly why this recipe prioritises review rather than claims to replace it.

The real queue is built once here, from every `validation` and `test` patch `review_priority` sends to review -- `demo` patches are excluded, since neither is ever reviewed and they were never scored in the first place. `ReviewQueue` has no reordering of its own, so the full list is sorted once by `helpers.review_sort_key` before any of it is submitted, which is what makes the result globally ordered rather than the four-patch preview's own approximation above.

In [ ]:
to_review = [
    (e.id, e, test_results[e.id] if e.split == "test" else val_results[e.id])
    for e in val_examples + test_examples
]
to_review = [row for row in to_review if row[2].priority == helpers.REVIEW]
to_review.sort(key=lambda row: (helpers.review_sort_key(row[2]), row[0]))

queue = ReviewQueue()
for example_id, example, result in to_review:
    item = helpers.queue_item(example_id, example.fields, result)
    gating = helpers.gating_label(result)
    queue.submit(item, result.reason, answer=decide(example)[gating])

print(f"queue holds {len(queue)} patches, least confident first{check}")
for row in queue.to_dicts():
    print(f"  {row['item']['id']}: {row['reason']}")

## What was and was not measured

Not measured live: the offline run replays synthetic answers written for this recipe, so its numbers check the pipeline and say nothing about how Jev performs. A deterministic review priority built on top of these three propositions is only as good as they are: the three confidently-wrong patches shown above, `t17`, `t18` and `t19`, each describe a real concern next to a comment that claims a safeguard the diff does not actually have -- the same shape of mistake this recipe's own fixtures were written to exercise, not a general claim about how often real patches would fool it.

**This recipe prioritises which patches a human reviewer looks at first. It is not a security scanner, and nothing here approves, blocks, merges, or deploys anything.** Every review priority is a label this recipe attaches to a patch for a human to act on; a `routine` priority means none of these three concerns was flagged confidently enough to say otherwise, not that the patch is safe, and acting on any priority is left entirely to whatever system or person calls this rule.

In [ ]:
if offline:
    print(f"Provenance: {backend.mode}. Not measured live: a pipeline check, not a Jev result.")
    print(f"N: {len(val_examples)} validation and {len(test_examples)} test patches")
else:
    dates = ", ".join(getattr(backend, "recorded_dates", ()) or ()) or "this run"
    print(f"Provenance: {backend.mode}, model {backend.model}")
    print(f"Captured: {dates}")
    print(f"N: {len(val_examples)} validation and {len(test_examples)} test patches")

## Next steps

- Add a harder lookalike to `RAW_ROWS` in `build_fixtures.py` -- a patch that claims a safeguard in two places at once (a comment and a misleading commit title) -- and see which threshold or the gate it lands on.
- Loosen the confidence gate (a lower `min_coverage`, passed to `helpers.choose_gate`) and watch `evaluate_selective`'s coverage rise on `test` while its risk rises too; `t17`, `t18` and `t19`, the patches named above, never move under a looser gate, because loosening only accepts more, never fewer, and each of their confidences already sits far above the current gate (0.68).
- Neighbouring recipes: [`25-change-evidence-review`](../25-change-evidence-review/) (the same three-independent-`Noul` composition pattern, in software engineering's neighbouring `Noul` recipe) and [`19-ci-failure-classification`](../19-ci-failure-classification/) (a `Choice` question, also software engineering).